# We use the 300 csv files created and implement a direct contest against the r codes.

In [1]:
%run -i ~/project/preambles
%run -i ~/project/helper_functions
%run -i ~/project/fitting_functions

In [2]:
# # Global parameters:
number_of_cycles = 500 # how many passes through the training data we go through
number_of_groups = 1 # divide the data set into smaller ones, to make fitting easier.
locations_per_group = 200 # how many locations to observe per group
number_of_locations = number_of_groups * locations_per_group # total locations
number_of_simulations = 300 # for synthetic data, how many optimisation to average over
steps_per_batch = 5
dims = 2  # 2D spatial
p =  3 # how many features 

In [3]:
X,Y,true_K, alpha_matrix_true, nu_matrix_true, sigma_matrix_true = Genton_parametrisation(number_of_locations, dims)
ground_truth_df = store_as_df(alpha_matrix_true, nu_matrix_true, sigma_matrix_true)
ground_truth_df

,alpha_matrix_11,nu_matrix_11,sigma_matrix_11,alpha_matrix_12,nu_matrix_12,sigma_matrix_12,alpha_matrix_13,nu_matrix_13,sigma_matrix_13,alpha_matrix_22,nu_matrix_22,sigma_matrix_22,alpha_matrix_23,nu_matrix_23,sigma_matrix_23,alpha_matrix_33,nu_matrix_33,sigma_matrix_33
0,0.01,1.2,1.0,0.0205,1.093,-0.286,0.0263,1.092,-0.181,0.02,0.6,1.0,0.0282,0.99,0.274,0.03,0.3,1.0


In [4]:
estimated_params_df = pd.DataFrame()
distance_K_df = pd.DataFrame()
for _ in range(1,301):
    print(f"data set {_}")
    try:
        # Load the CSV file
        df = pd.read_csv(f'~/project/synthetic_data/realisation_{_}.csv')
        # Extract the spatial coordinates (first and second spatial coordinates)
        X = torch.tensor(df[['0','1']].values[:200], dtype=torch.float64).detach()
        # Prepare Y by reshaping the gene expression levels for each gene
        expression = df['Expression'].values.reshape(200, 3)
        Y = torch.tensor(expression, dtype=torch.float64).detach()
        torch.autograd.set_detect_anomaly(True)
        optimized_marginal_params = optimize_marginal_parameters(X, Y, number_of_groups,  number_of_cycles, steps_per_batch)
        alpha_matrix, nu_matrix, sigma_matrix = optimize_cross_parameters(optimized_marginal_params,X,Y,number_of_groups,number_of_cycles,steps_per_batch)
        estimated_K = compute_matern_covariance(alpha_matrix, nu_matrix, sigma_matrix, X)
        distance_K = torch.norm(true_K - estimated_K)**2 / torch.norm(true_K)**2
        distance_K_df = pd.concat([distance_K_df, pd.DataFrame([distance_K.item()])] ,ignore_index=True)
        estimated_params_df = pd.concat([estimated_params_df, store_as_df(alpha_matrix, nu_matrix, sigma_matrix) ], ignore_index=True)
        # Save the combined DataFrame to a CSV file
        file_path = os.path.expanduser(f'~/project/python_processed_data/synthetic_estimated_parameters_2.csv')
        estimated_params_df.to_csv(file_path, index=False)
    except Exception as e:
        print(e)
        continue
estimated_params_df

data set 1
the values are 1.000, 0.0100000000000000, 1.000
the gradients are 66663.734, 10.9703914524359742, 19.339
the values are 0.900, 0.0000000000000002, 0.900
the gradients are 0.000, 0.0000000000000000, 12.308
the values are 0.833, 0.0000000000000002, 0.804
the gradients are 0.000, 0.0000000000000000, 0.571
the values are 0.781, 0.0000000000000002, 0.727
the gradients are 0.000, 0.0000000000000000, -13.626
the values are 0.739, 0.0000000000000002, 0.703
the gradients are 0.000, 0.0000000000000000, -19.371
the values are 0.703, 0.0000000000000002, 0.719
the gradients are 0.000, 0.0000000000000000, -15.502
the values are 0.672, 0.0000000000000002, 0.754
the gradients are 0.000, 0.0000000000000000, -8.067
the values are 0.645, 0.0000000000000002, 0.794
the gradients are 0.000, 0.0000000000000000, -0.892
the values are 0.621, 0.0000000000000002, 0.831
the gradients are 0.000, 0.0000000000000000, 4.488
the values are 0.600, 0.0000000000000002, 0.858
the gradients are 0.000, 0.00000000

KeyboardInterrupt: 

In [ ]:
histograms_are_plotted = True
df_to_plot = estimated_params_df
%run -i epilogue

In [ ]:
plt.hist(distance_K_df[0])

In [1]:
# After fitting is successful, we are writing down the summary statistics for the various variables. 

In [16]:
import pandas as pd

# Assuming df is your DataFrame with the data, and ground_truth_df stores the true values

# Calculate summary statistics: min, Q1, median, mean, Q3, max for each variable
summary_table = df.describe(percentiles=[0.25, 0.5, 0.75]).loc[['min', '25%', '50%', 'mean', '75%', 'max']]

# Adding the ground truth values (assuming ground_truth_df contains the true values as a single row DataFrame)
truths = pd.DataFrame(ground_truth_df.loc[0]).T
truths.index = ['true']

# Concatenate the summary statistics with the true values
comparison_table = pd.concat([truths, summary_table])

# Format the table to show 4 decimal places
comparison_table = comparison_table.map(lambda x: f'{x:.4f}' if isinstance(x, (int, float)) else x)
# Display the table
comparison_table.T

,true,min,25%,50%,mean,75%,max
alpha_matrix_11,0.0100,0.0000,0.0000,0.0059,0.0169,0.0184,0.3155
nu_matrix_11,1.2000,0.0046,0.9805,1.0177,1.5516,2.0692,4.0056
sigma_matrix_11,1.0000,0.7302,0.9259,0.9877,0.9894,1.0467,1.2536
alpha_matrix_12,0.0205,0.2111,0.9000,0.9044,0.9141,0.9319,1.8288
nu_matrix_12,1.0930,0.0296,1.7924,2.3676,2.5579,3.2452,7.5811
sigma_matrix_12,-0.2860,-0.0713,-0.0000,-0.0000,-0.0019,-0.0000,0.0000
alpha_matrix_13,0.0263,0.2172,0.9000,0.9044,0.9142,0.9325,1.8289
nu_matrix_13,1.0920,0.0935,1.5591,2.0423,2.4547,3.2014,7.7109
sigma_matrix_13,-0.1810,-0.1083,-0.0000,-0.0000,-0.0030,-0.0000,0.0000
alpha_matrix_22,0.0200,0.0000,0.0000,0.0050,0.0141,0.0186,0.3016
